# Persona Vectors: Predicting Fine-Tuning Shift

This notebook validates the paper's training-data-screening claim: **projecting a
fine-tuning dataset onto a persona vector, before ever fine-tuning on it, predicts how
much that data will actually shift the model's behavior afterward.**

Every other notebook in this repo demonstrates persona vectors as an *inference-time*
tool (steering, monitoring a frozen model). This one is different: it uses the vector to
predict a *training-time* outcome, then actually fine-tunes and checks the prediction.

Reuses real infrastructure from the cloned `persona_vectors` repository
(`Claude/persona_vectors/`):
- `dataset.zip` → three real severity levels of "evil"-trait training data
  (`dataset/evil/{normal,misaligned_1,misaligned_2}.jsonl`)
- `data_generation/trait_data_extract/evil.json` → real trait instructions
  (word-for-word identical to what earlier notebooks in this repo already hardcode)
- `data_generation/trait_data_eval/evil.json` → 20 real held-out eval questions
- `sft.py`'s `sft_train`, `validate.py`'s `TrainingConfig` → real LoRA fine-tuning code,
  via `unsloth`

**Model**: Qwen/Qwen2.5-7B-Instruct

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
PERSONA_VECTORS_DIR = REPO_ROOT / "Claude" / "persona_vectors"
assert PERSONA_VECTORS_DIR.exists(), f"Expected cloned repo at {PERSONA_VECTORS_DIR}"
sys.path.insert(0, str(PERSONA_VECTORS_DIR))

from unsloth import FastLanguageModel  # must import before torch/transformers

import os
import json
import zipfile
import random
import time

import torch
import torch.nn.functional as F
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
from datasets import Dataset

from sft import sft_train
from validate import TrainingConfig

torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")
print("Imported sft_train, TrainingConfig, FastLanguageModel from the real persona_vectors repo.")

In [ ]:
DATASET_DIR = PERSONA_VECTORS_DIR / "dataset"
if not DATASET_DIR.exists():
    print("Extracting dataset.zip...")
    with zipfile.ZipFile(PERSONA_VECTORS_DIR / "dataset.zip") as zf:
        zf.extractall(PERSONA_VECTORS_DIR)
    print("Done.")
else:
    print("dataset/ already extracted.")

EVIL_DATASET_DIR = DATASET_DIR / "evil"
SEVERITY_FILES = {
    "normal": EVIL_DATASET_DIR / "normal.jsonl",
    "misaligned_1": EVIL_DATASET_DIR / "misaligned_1.jsonl",
    "misaligned_2": EVIL_DATASET_DIR / "misaligned_2.jsonl",
}
for name, path in SEVERITY_FILES.items():
    assert path.exists(), f"Missing {path}"
    with open(path) as f:
        n_lines = sum(1 for _ in f)
    print(f"{name}: {path.name} ({n_lines} examples)")

In [ ]:
with open(PERSONA_VECTORS_DIR / "data_generation" / "trait_data_extract" / "evil.json") as f:
    evil_extract_data = json.load(f)

with open(PERSONA_VECTORS_DIR / "data_generation" / "trait_data_eval" / "evil.json") as f:
    evil_eval_data = json.load(f)

EVIL_POS_INSTRUCTION = evil_extract_data["instruction"][0]["pos"]
EVIL_NEG_INSTRUCTION = evil_extract_data["instruction"][0]["neg"]
EXTRACTION_QUESTIONS = evil_extract_data["questions"]
EVAL_QUESTIONS = evil_eval_data["questions"]

print(f"Positive instruction: {EVIL_POS_INSTRUCTION}")
print(f"Negative instruction: {EVIL_NEG_INSTRUCTION}")
print(f"Extraction questions: {len(EXTRACTION_QUESTIONS)}")
print(f"Eval questions: {len(EVAL_QUESTIONS)}")
print("\nSample eval questions:")
for q in EVAL_QUESTIONS[:3]:
    print(f"  - {q}")